In [ ]:
# Import python libraries

In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import os

In [4]:
from dateutil import parser as dparser

In [5]:
# Load all 5 tables

In [6]:
fact = pd.read_csv('D:/Portfolio/Projects/retail_portfolio/data/raw/fact_orders.csv')

In [7]:
dim_c = pd.read_csv('D:/Portfolio/Projects/retail_portfolio/data/raw/dim_customers.csv')

In [8]:
dim_p = pd.read_csv('D:/Portfolio/Projects/retail_portfolio/data/raw/dim_products.csv')

In [9]:
dim_l = pd.read_csv('D:/Portfolio/Projects/retail_portfolio/data/raw/dim_locations.csv')

In [10]:
dim_r = pd.read_csv('D:/Portfolio/Projects/retail_portfolio/data/raw/dim_sales_reps.csv')

In [11]:
# check dimension of datasets

In [12]:
print("fact_orders shape: ", fact.shape)

fact_orders shape:  (5204, 16)


In [13]:
print("dim_customers shape: ", dim_c.shape)

dim_customers shape:  (1, 11)


In [14]:
print("dim_products shape:  ", dim_p.shape)

dim_products shape:   (1, 8)


In [15]:
print("dim_locations shape: ", dim_l.shape)

dim_locations shape:  (104, 7)


In [16]:
print("dim_sales_reps shape:", dim_r.shape)

dim_sales_reps shape: (78, 7)


In [17]:
# Data Quality Report - Before Cleaning

In [20]:
for name, df in [('fact_orders', fact), ('dim_customers', dim_c),
                ('dim_products', dim_p), ('dim_location', dim_l),
                ('dim_sales_reps', dim_r)]:
    print(f"\n{'='*50}")
    print(f"TABLE: {name}")
    print(f"Shape: {df.shape}")
    print(f"Nulls  per column:\n{df.isnull().sum()}")
    print(f"Exact duplicates: {df.duplicated().sum()}")
    print(f"Data types:\n{df.dtypes}")


TABLE: fact_orders
Shape: (5204, 16)
Nulls  per column:
order_id                    0
customer_id                 0
product_id                  0
location_id               570
rep_id                    623
order_date                  0
ship_date                   0
quantity                    0
unit_price                  0
revenue                     0
cost                        0
discount_pct              519
status                      0
region                    676
custmer_id               4996
_region_mismatch_flag    5198
dtype: int64
Exact duplicates: 0
Data types:
order_id                  object
customer_id               object
product_id                object
location_id               object
rep_id                    object
order_date                object
ship_date                 object
quantity                   int64
unit_price                object
revenue                  float64
cost                     float64
discount_pct             float64
status                

In [23]:
# Standardise all ID prefixes (All tables)
# must be done before any joins or checks

In [24]:
import re

In [26]:
def normalise_id(series, prefix):
    # Extract the numeric part, zero-pad to 4 digitd, re-apply prefix
    nums = series.astype(str).str.extract(r'(\d+)')[0]
    return nums.str.zfill(4).apply(lambda x: f'{prefix}-{x}')

fact['customer_id'] = normalise_id(fact['customer_id'], 'CUST')
dim_c['cust_id'] = normalise_id(dim_c['cust_id'], 'CUST')
fact['product_id'] = normalise_id(fact['product_id'], 'PROD')
dim_p['product_id'] = normalise_id(dim_p['product_id'], 'PROD')
fact['location_id'] = normalise_id(fact['location_id'], 'LOC')
dim_l['location_id'] = normalise_id(dim_l['location_id'], 'LOC')
fact['rep_id'] = normalise_id(fact['rep_id'], 'REP')
dim_r['rep_id'] = normalise_id(dim_r['rep_id'], 'REP')

print("✅ ID prefixes standardised across all tables")

✅ ID prefixes standardised across all tables


In [27]:
# Remove duplicates

In [28]:
for name, df in [('fact_orders', fact), ('dim_customers', dim_c),
                ('dim_products', dim_p), ('dim_location', dim_l),
                ('dim_sales_reps', dim_r)]:
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f"{name}: removed {before - len(df)} exact duplicate rows")
    
# Near duplicate order_id (e.g. 'ord-0001' & 'ORD-0001')
fact['order_id'] = fact['order_id'].str.upper()
before = len(fact)
fact.drop_duplicates(subset=['order_id'], keep='first', inplace=True)
print(f"fact_orders: removed {before - len(fact)} near-duplicate order_id rows")

# Near duplicate customer names
dim_c['full_name'] = (dim_c['full_name']
                     .str.strip()
                     .str.lower()
                     .str.replace(r'\s+', ' ', regex=True)
                     .str.title())
print("✅ Duplicates removed from all tables")

fact_orders: removed 0 exact duplicate rows
dim_customers: removed 0 exact duplicate rows
dim_products: removed 0 exact duplicate rows
dim_location: removed 0 exact duplicate rows
dim_sales_reps: removed 0 exact duplicate rows
fact_orders: removed 204 near-duplicate order_id rows
✅ Duplicates removed from all tables


In [29]:
# Standardise date formats
# Every date column contains three mixed formats: MM/DD/YYYY, YYYY-MM-DD, and DD-Mon-YYYY

In [33]:
def safe_parse_date(val):
    try:
        return pd.Timestamp(dparser.parse(str(val), dayfirst=False))
    except Exception:
        return pd.NaT
    
# fact_orders date
for col in ['order_date', 'ship_date']:
    fact[col] = fact[col].apply(safe_parse_date)
    failed = fact[col].isna().sum()
    print(f"fact_orders.{col}: {failed} rows failed to parse → set to NaT")
    
# dimension table dates
dim_c['join_date'] = dim_c['join_date'].apply(safe_parse_date)
dim_p['launch_date'] = dim_p['launch_date'].apply(safe_parse_date)
dim_r['hire_date'] = dim_r['hire_date'].apply(safe_parse_date)
dim_r['end_date'] = dim_r['end_date'].apply(safe_parse_date)

print("✅ all dates columns standardised")
        

fact_orders.order_date: 0 rows failed to parse → set to NaT
fact_orders.ship_date: 0 rows failed to parse → set to NaT
✅ all dates columns standardised


In [34]:
# Parse currency and numeric fields

In [36]:
def parse_currency(val):
    if pd.isna(val):
        return np.nan
    return float(str(val).replace('$', '').replace(',', '').strip())

fact['unit_price'] = fact['unit_price'].apply(parse_currency)
fact['revenue'] = fact['revenue'].apply(parse_currency)
fact['cost'] = fact['cost'].apply(parse_currency)
dim_p['unit_price'] = dim_p['unit_price'].apply(parse_currency)
dim_p['unit_cost'] = dim_p['unit_cost'].apply(parse_currency)

# Nullify negative values and log them
neg_cost = dim_p[dim_p['unit_cost'] < 0].shape[0]
neg_rev = fact[fact['revenue'] < 0].shape[0]

dim_p.loc[dim_p['unit_cost'] < 0, 'unit_cost'] = np.nan
fact.loc[fact['revenue'] < 0, 'revenue'] = np.nan

print(f"Nullified {neg_cost} negative unit_cost rows in dim_products")
print(f"Nullified {neg_rev} negative revenue rows in fact_orders")
print("✅ Currency fields cleaned")

Nullified 0 negative unit_cost rows in dim_products
Nullified 51 negative revenue rows in fact_orders
✅ Currency fields cleaned


In [37]:
# Standardise phone and boolean fields

In [38]:
# Phone: keep digits only, max 10 characters
dim_c ['phone'] = (dim_c['phone']
                   .astype(str)
                   .str.replace(r'\D', '', regex=True)
                   .str.slice(0, 10))

# Boolean: convert True/Yes/1/False/No/0 → 1 or 0
BOOL_MAP = {'true':1, 'yes':1, '1':1, 'false':0, 'no':0, '0':0}

for df_, col in [(dim_c, 'is_active'), (dim_p, 'is_active'), (dim_r, 'is_active')]:
    df_[col] = df_[col].astype(str).str.strip().str.lower().map(BOOL_MAP)
    
print("✅ phone and boolean fields standardised")

✅ phone and boolean fields standardised


In [39]:
# Handle missing values

In [41]:
# Foreign key nulls → fill with UNKNOWN sentinel
for col in ['rep_id', 'location_id']:
    nulls = fact[col].isna().sum()
    fact[col] = fact[col].fillna('UNKNOWN')
    print(f"fact_orders.{col}: filled {nulls} nulls with UNKNOWN")
    
# Numeric → fill discount_pct with 0 (no discount assumed)
fact['discount_pct'] = fact['discount_pct'].fillna(0)

# Categorical → fill region with Unknown
fact['region'] = fact['region'].fillna('Unknown')

# Primary key nulls → drop (record is unusable)
before = len(fact)
fact.dropna(subset=['order_id'], inplace=True)
print(f"Dropped {before - len(fact)} rows with null order_id")

# PII columns (email, phone) → leave as NaN, do not impute
print("✅ Missing values handled")


fact_orders.rep_id: filled 0 nulls with UNKNOWN
fact_orders.location_id: filled 0 nulls with UNKNOWN
Dropped 0 rows with null order_id
✅ Missing values handled


In [42]:
# Referential integrity check

In [52]:
# Define quarantine path
quarantine_path = "D:/Portfolio/Projects/retail_portfolio/data/quarantine"
os.makedirs(quarantine_path, exist_ok=True)

In [53]:
# After renaming, rename dim_c column to match fact
dim_c.rename(columns={'cust_id': 'customer_id'}, inplace=True)

# Check each FK
checks = {
    'customer_id': dim_c['customer_id'],
    'product_id': dim_p['product_id'],
    'location_id': dim_l['location_id'],
    'rep_id': dim_r['rep_id'],
}

fact_clean = fact.copy()

for fk_col, valid_keys in checks.items():
    valid_set = set(valid_keys)
    orphans = fact_clean[~fact_clean[fk_col].isin(valid_set) &
                        (fact_clean[fk_col] != 'UNKNOWN')]
    print(f"Orphaned {fk_col}: {len(orphans)} rows → saved to quarantine")
    if len(orphans) > 0:
        orphans.assign(error=f'ORPHANED_{fk_col.upper()}') \
               .to_csv(f"{quarantine_path}/orphaned_{fk_col}.csv", index=False)
    fact_clean = fact_clean[fact_clean[fk_col].isin(valid_set) |
                            (fact_clean[fk_col] == 'UNKNOWN')].copy()
    
print(f"\nfact_orders rows before: {len(fact)}")
print(f"fact_orders rows after: {len(fact_clean)}")
print("✅ Referential integrity check complete")

Orphaned customer_id: 4998 rows → saved to quarantine
Orphaned product_id: 2 rows → saved to quarantine
Orphaned location_id: 0 rows → saved to quarantine
Orphaned rep_id: 0 rows → saved to quarantine

fact_orders rows before: 5000
fact_orders rows after: 0
✅ Referential integrity check complete


In [50]:
# Save all cleaned CSVs

In [54]:
# Define Windows folder path
cleaned_path = "D:/Portfolio/Projects/retail_portfolio/data/cleaned"
os.makedirs(cleaned_path, exist_ok=True)

In [57]:
dim_c_clean = dim_c.copy()
dim_p_clean = dim_p.copy()
dim_l_clean = dim_l.copy()
dim_r_clean = dim_r.copy()

tables = {
    'fact_orders': fact_clean,
    'dim_customers': dim_c_clean,
    'dim_products': dim_p_clean,
    'dim_locations': dim_l_clean,
    'dim_sales_reps': dim_r_clean,
}

for name, df in tables.items():
    df.to_csv(f"{cleaned_path}/{name}.csv", index = False)
    print(f"Saved {name}: {df.shape}")

print("\n✅ All cleaned files saved to cleaned folder")

Saved fact_orders: (0, 16)
Saved dim_customers: (1, 11)
Saved dim_products: (1, 8)
Saved dim_locations: (104, 7)
Saved dim_sales_reps: (78, 7)

✅ All cleaned files saved to cleaned folder


In [58]:
# Data quality report (after cleaning)

In [59]:
for name, df in tables.items():
    print(f"\n{'='*50}")
    print(f"TABLE: {name}")
    print(f"Shape: {df.shape}")
    print(f"Remaining nulls:\n{df.isnull().sum()}")
    print(f"Remaining duplicates: {df.duplicated().sum()}")


TABLE: fact_orders
Shape: (0, 16)
Remaining nulls:
order_id                 0.0
customer_id              0.0
product_id               0.0
location_id              0.0
rep_id                   0.0
order_date               0.0
ship_date                0.0
quantity                 0.0
unit_price               0.0
revenue                  0.0
cost                     0.0
discount_pct             0.0
status                   0.0
region                   0.0
custmer_id               0.0
_region_mismatch_flag    0.0
dtype: float64
Remaining duplicates: 0

TABLE: dim_customers
Shape: (1, 11)
Remaining nulls:
customer_id     0
full_name       0
email           0
phone           0
region          0
state           0
segment         0
join_date       0
status          0
is_active       0
customer_age    0
dtype: int64
Remaining duplicates: 0

TABLE: dim_products
Shape: (1, 8)
Remaining nulls:
product_id      0
product_name    0
category        0
subcategory     0
unit_price      0
unit_cost     